# Issue #5: YAP vs HebPipe Segmenter Comparison

**Expected runtime:** ~20 minutes (mostly YAP lexicon load).

1. Clone repo (private — needs GitHub PAT with `repo` scope)
2. Build & run YAP (GOPATH mode, vendored deps, binary inside repo dir)
3. Run HebPipe's segmenter (RFTokenizer — the full hebpipe package is broken on modern stacks)
4. Compare & download results

## 1. Clone repo

In [ ]:
from getpass import getpass
token = getpass('GitHub PAT: ')
%cd /content
!rm -rf /content/NLP-Final-
!git clone https://{token}@github.com/AdonZahavi/NLP-Final-.git /content/NLP-Final-
%cd /content/NLP-Final-
!git checkout issue-5-segmenter-comparison
!wc -l segmentation/sample_50.txt

## 2. Build YAP

YAP predates Go modules: it needs GOPATH mode (`GO111MODULE=off`), its repo at `$GOPATH/src/yap`
(imports are `yap/app`, `yap/webapi`), and its `vendor/` dir — which has the correct old
`gonuts` packages but is missing `gorilla/mux` and `yaml.v2`, so we clone those two in.
The binary is built **inside the repo dir** so YAP's file search (`.`, `data/bgulex`) resolves
whether it looks relative to the cwd or to the executable.

In [ ]:
# Install Go and bzip2 (Colab's apt Go is fine for GOPATH mode)
!apt-get update -qq && apt-get install -y -qq golang-go bzip2 > /dev/null 2>&1
!go version

In [ ]:
# Clone YAP into GOPATH, complete vendor/, and build (all-in-one, safe to re-run)
import os
os.environ['GOPATH'] = '/content/gopath'
os.environ['GO111MODULE'] = 'off'

%cd /content
!rm -rf /content/gopath /content/yap_bin
!mkdir -p /content/gopath/src
!git clone https://github.com/OnlpLab/yap.git /content/gopath/src/yap
%cd /content/gopath/src/yap
!bunzip2 -k data/*.bz2

# vendor/ lacks these two packages — add them directly (no go.mod, no version resolution)
!git clone -q --depth 1 https://github.com/gorilla/mux.git vendor/github.com/gorilla/mux
!rm -rf vendor/github.com/gorilla/mux/.git
!mkdir -p vendor/gopkg.in
!git clone -q --depth 1 --branch v2 https://github.com/go-yaml/yaml.git vendor/gopkg.in/yaml.v2
!rm -rf vendor/gopkg.in/yaml.v2/.git

# Lexicon symlinks for the '.' search path
!ln -sf data/bgulex/bgupreflex_withdef.utf8.hr .
!ln -sf data/bgulex/bgulex.utf8.hr .

# Build INTO the repo dir: executable-dir == cwd == repo root
!go build -o /content/gopath/src/yap/yap_bin .
!ls -la /content/gopath/src/yap/yap_bin

In [ ]:
# Start YAP API server — log to FILE (an undrained PIPE freezes YAP mid-run), poll with liveness check
import subprocess, time, urllib.request, json

logf = open('/content/yap.log', 'w')
yap_proc = subprocess.Popen(
    ['./yap_bin', 'api'],
    cwd='/content/gopath/src/yap',
    stdout=logf, stderr=subprocess.STDOUT,
)

req = urllib.request.Request(
    'http://localhost:8000/yap/heb/joint',
    data=json.dumps({'text': 'שלום  '}).encode(),
    headers={'Content-Type': 'application/json'},
)
for attempt in range(60):
    if yap_proc.poll() is not None:
        print(f'YAP EXITED (code {yap_proc.returncode}). Log tail:')
        print(open('/content/yap.log').read()[-3000:])
        break
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            print(f'YAP ready (attempt {attempt+1}):', list(json.loads(r.read()).keys()))
            break
    except Exception as e:
        print(f'  waiting... ({attempt+1}/60, {type(e).__name__})')
        time.sleep(10)
else:
    print('Timed out. Log tail:')
    print(open('/content/yap.log').read()[-3000:])

In [ ]:
# Run YAP on sample_50.txt (a few minutes — joint parsing is slow)
%cd /content/NLP-Final-
!python segmentation/yap_client.py --host http://localhost:8000

In [ ]:
# Stop YAP
yap_proc.terminate()
yap_proc.wait()
logf.close()
print('YAP server stopped.')

## 3. Run HebPipe's segmenter (RFTokenizer)

The full `hebpipe` package fails at import time on Colab's modern stack (its
xrenner/coref module), needs an interactive model-download prompt, and runs a
Stanza tagging pass even for segmentation-only jobs. Its actual segmentation
engine is **RFTokenizer** (same author, same `heb.sm3` model — `heb_pipe.py -wt`
just calls `rf_tokenize`), so we run that component directly.

In [ ]:
!pip install -q rftokenizer

In [ ]:
%%writefile /content/NLP-Final-/segmentation/hebpipe_runner.py
"""Segment sample_50.txt with RFTokenizer — HebPipe's morphological segmenter."""

from __future__ import annotations

import json
import re
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
IN_TXT = ROOT / "segmentation" / "sample_50.txt"
OUT = ROOT / "segmentation" / "out_hebpipe.jsonl"
MODEL = ROOT / "segmentation" / "heb.sm3"
# same model file the hebpipe package downloads for itself
MODEL_URL = "http://gucorpling.org/amir/download/heb_models_v4/heb.sm3"

# same minimal tokenization as yap_client.py, so both tools see identical tokens
TOKENIZE_RE = re.compile(r"[א-ת0-9a-zA-Z\"'׳״]+|[^\s]")


def main() -> None:
    if not MODEL.exists():
        print(f"downloading {MODEL_URL} …")
        urllib.request.urlretrieve(MODEL_URL, MODEL)

    from rftokenizer import RFTokenizer

    tok = RFTokenizer(model=str(MODEL))
    sentences = IN_TXT.read_text(encoding="utf-8").strip().splitlines()
    with open(OUT, "w", encoding="utf-8", newline="\n") as f:
        for i, sent in enumerate(sentences, 1):
            words = TOKENIZE_RE.findall(sent)
            segged = tok.rf_tokenize(words)
            tokens = [
                {"token": w, "morphemes": s.split("|")}
                for w, s in zip(words, segged)
            ]
            f.write(json.dumps({"text": sent, "tokens": tokens}, ensure_ascii=False) + "\n")
            print(f"  [{i}/{len(sentences)}] ok")
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()

In [ ]:
# Downloads heb.sm3 (~a few MB) on first run
%cd /content/NLP-Final-
!python segmentation/hebpipe_runner.py

## 4. Compare and download

In [ ]:
%cd /content/NLP-Final-
!python segmentation/compare_segmenters.py

In [ ]:
!cat segmentation/comparison.md

In [ ]:
from google.colab import files
for f in ['segmentation/out_yap.jsonl', 'segmentation/out_hebpipe.jsonl', 'segmentation/comparison.md']:
    files.download(f'/content/NLP-Final-/{f}')
    print(f'downloaded {f}')